# Giáo trình Dữ liệu lớn – Chương 4

Notebook tổng hợp các đoạn mã trong chương, chạy trên **Google Colab** (ô đầu cài OpenJDK 17, PySpark 3.5.7 và tải kho mã). Trên **Databricks Free Edition**: bỏ ô cài đặt, tải thư mục `data/` lên volume `/Volumes/workspace/default/du_lieu/` do người học tự tạo và thay `data/` bằng đường dẫn này; tính toán serverless của Free Edition không hỗ trợ API RDD/`SparkContext` và `cache()`/`persist()` (xem [README](https://github.com/mocminh/bigdata_code#databricks-free-edition)).

Khác biệt so với sách: đường dẫn `hdfs://.../data/` được đổi thành `data/`, thư mục ghi kết quả là `output/` và `models/`; lệnh `spark.stop()` được đổi thành ghi chú để các ô sau vẫn chạy được. Mã nguyên văn: `code/ch04/doan_ma_*.py`.


In [ ]:
# --- Chuan bi moi truong (Google Colab) ---
# Buoc 1: cai OpenJDK 17 (Spark 3.5 ho tro Java 8/11/17)
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
# Buoc 2: cai dat PySpark tu PyPI (ghim phien ban theo Bang 2.3)
!pip install -q pyspark==3.5.7
if not os.path.exists("data"):
    !git clone -q https://github.com/mocminh/bigdata_code
    %cd bigdata_code
import shutil
for thu_muc in ("output", "models"):          # don ket qua cua lan chay truoc
    shutil.rmtree(thu_muc, ignore_errors=True)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (SparkSession.builder.master("local[*]")
         .appName("GiaoTrinhDuLieuLon-ch04").getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark", spark.version)

## Đoạn mã 4.1. Tạo DataFrame từ tập hợp cục bộ bằng createDataFrame.


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Chuong4-DataFrame") \
    .getOrCreate()

# Tao DataFrame tu danh sach tuple kem chuoi khai bao luoc do
du_lieu = [(1, "Lan", "Ha Noi", 28),
           (2, "Minh", "Da Nang", 35),
           (3, "Hoa", "Can Tho", 41)]
df_nv = spark.createDataFrame(
    du_lieu, schema="ma INT, ten STRING, tinh STRING, tuoi INT")
df_nv.show()
df_nv.printSchema()

## Đoạn mã 4.2. Đọc DataFrame từ các tệp CSV, JSON và Parquet.


In [ ]:
# Doc tep CSV co dong tieu de, tu suy dien kieu du lieu
df_csv = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv("data/donhang.csv"))

# Doc tep JSON va tep Parquet
df_json = spark.read.json("data/khachhang.json")
df_parquet = spark.read.parquet("data/giaodich.parquet")

## Đoạn mã 4.3. Tạo DataFrame từ RDD với lược đồ StructType tường minh.


In [ ]:
from pyspark.sql.types import (StructType, StructField,
                               StringType, IntegerType, DoubleType)

rdd = spark.sparkContext.parallelize(
    [("SP01", "Laptop", 25, 1500.0),
     ("SP02", "Ban phim", 120, 25.5)])

schema = StructType([
    StructField("ma_sp", StringType(), False),
    StructField("ten_sp", StringType(), True),
    StructField("ton_kho", IntegerType(), True),
    StructField("gia", DoubleType(), True)])

df_sp = spark.createDataFrame(rdd, schema)
df_sp.printSchema()

## Đoạn mã 4.4. Xem kế hoạch truy vấn bằng explain().


In [ ]:
df = spark.read.parquet("data/donhang.parquet")
ket_qua = (df.filter(df.khu_vuc == "Nam")
             .select("ma_don", "don_gia")
             .filter(df.don_gia > 100.0))

ket_qua.explain()                 # chi in ke hoach vat ly
ket_qua.explain(mode="extended")  # in cac ke hoach logic va vat ly
# Trich ke hoach vat ly (rut gon):
# *(1) Project [ma_don, don_gia]
# +- *(1) Filter (isnotnull(khu_vuc) AND isnotnull(don_gia)
#                 AND (khu_vuc = Nam) AND (don_gia > 100.0))
#    +- FileScan parquet [ma_don, don_gia, khu_vuc]
#       PushedFilters: [IsNotNull(khu_vuc), IsNotNull(don_gia),
#                       EqualTo(khu_vuc,Nam),
#                       GreaterThan(don_gia,100.0)]

## Đoạn mã 4.5. Khởi tạo tập dữ liệu bán hàng dùng chung cho cả chương.


In [ ]:
from pyspark.sql import functions as F

don_hang = [
    ("D001", "KH01", "Laptop",     "Dien tu",
     2, 1500.0, "2024-03-01", "Bac"),
    ("D002", "KH02", "Dien thoai", "Dien tu",
     5,  800.0, "2024-03-02", "Nam"),
    ("D003", "KH01", "Ao thun",    "Thoi trang",
     10,  12.5, "2024-03-02", "Bac"),
    ("D004", "KH03", "Tu lanh",    "Gia dung",
     1,  950.0, "2024-03-05", "Trung"),
    ("D005", "KH02", "Quan jean",  "Thoi trang",
     3,   45.0, "2024-03-06", "Nam"),
    ("D006", "KH04", "May giat",   "Gia dung",
     2,  700.0, None, "Nam")]

cot = ("ma_don STRING, ma_kh STRING, san_pham STRING, "
       "danh_muc STRING, so_luong INT, don_gia DOUBLE, "
       "ngay_dat STRING, khu_vuc STRING")
df = spark.createDataFrame(don_hang, schema=cot)
df.show()

## Đoạn mã 4.6. Các thao tác select, withColumn, filter/where và orderBy.


In [ ]:
# select: chon mot so cot can quan tam
df.select("ma_don", "san_pham", "don_gia").show(3)

# withColumn: them cot thanh_tien = so_luong * don_gia
df2 = df.withColumn("thanh_tien",
                    F.col("so_luong") * F.col("don_gia"))

# filter va where la hai cach viet tuong duong
df2.filter(F.col("thanh_tien") > 1000).show()
df2.where("danh_muc = 'Dien tu' AND so_luong >= 2").show()

# orderBy: sap xep giam dan theo thanh_tien
df2.orderBy(F.col("thanh_tien").desc()).show(5)

## Đoạn mã 4.7. Tổng hợp doanh thu theo danh mục với groupBy và agg.


In [ ]:
bao_cao = (df2.groupBy("danh_muc")
    .agg(F.count("ma_don").alias("so_don"),
         F.sum("thanh_tien").alias("tong_doanh_thu"),
         F.avg("thanh_tien").alias("doanh_thu_tb"),
         F.min("don_gia").alias("gia_thap_nhat"),
         F.max("don_gia").alias("gia_cao_nhat"))
    .orderBy(F.col("tong_doanh_thu").desc()))
bao_cao.show()

## Đoạn mã 4.8. Xử lý giá trị thiếu bằng dropna và fillna.


In [ ]:
# Loai bo cac dong thieu ngay_dat
df_sach = df2.dropna(subset=["ngay_dat"])

# Thay the gia tri thieu theo tung cot
df_thay = df2.fillna({"ngay_dat": "1900-01-01", "so_luong": 0})

## Đoạn mã 4.9. Các loại join giữa bảng đơn hàng và bảng khách hàng.


In [ ]:
khach_hang = spark.createDataFrame(
    [("KH01", "Nguyen Van An", "Ha Noi"),
     ("KH02", "Tran Thi Binh", "TP HCM"),
     ("KH05", "Le Van Cuong",  "Hue")],
    schema="ma_kh STRING, ten_kh STRING, thanh_pho STRING")

df2.join(khach_hang, on="ma_kh", how="inner").show()
df2.join(khach_hang, on="ma_kh", how="left").show()
df2.join(khach_hang, on="ma_kh", how="full_outer").show()

# Broadcast join: phat tan bang nho toi moi executor
from pyspark.sql.functions import broadcast
df2.join(broadcast(khach_hang), on="ma_kh", how="inner").show()

## Đoạn mã 4.10. Đăng ký khung nhìn tạm và truy vấn bằng spark.sql.


In [ ]:
df2.createOrReplaceTempView("don_hang")

doanh_thu = spark.sql("""
    SELECT danh_muc,
           COUNT(ma_don)             AS so_don,
           SUM(thanh_tien)           AS tong_doanh_thu,
           ROUND(AVG(thanh_tien), 2) AS doanh_thu_tb
    FROM don_hang
    WHERE khu_vuc <> 'Trung'
    GROUP BY danh_muc
    HAVING SUM(thanh_tien) > 500
    ORDER BY tong_doanh_thu DESC
""")
doanh_thu.show()

## Đoạn mã 4.11. Khung nhìn tạm toàn cục và truy vấn join bằng SQL.


In [ ]:
khach_hang.createGlobalTempView("khach_hang")

spark.sql("""
    SELECT d.ma_don, k.ten_kh, d.san_pham, d.thanh_tien
    FROM don_hang d
    JOIN global_temp.khach_hang k ON d.ma_kh = k.ma_kh
    ORDER BY d.thanh_tien DESC
""").show()

## Đoạn mã 4.12. Ghi và đọc lại bảng Parquet có phân vùng theo khu vực.


In [ ]:
(df2.write
    .mode("overwrite")
    .partitionBy("khu_vuc")
    .parquet("output/warehouse/don_hang_parquet"))

# Doc lai: Spark chi quet thu muc phan vung thoa dieu kien loc
df_nam = (spark.read.parquet("output/warehouse/don_hang_parquet")
          .filter(F.col("khu_vuc") == "Nam"))
df_nam.show()

## Đoạn mã 4.13. Định nghĩa và đăng ký UDF cho DataFrame API và cho SQL.


In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def phan_loai(gia_tri):
    if gia_tri is None:
        return "Khong xac dinh"
    if gia_tri >= 1000:
        return "Don lon"
    return "Don thuong"

# Dang ky cho DataFrame API
phan_loai_udf = udf(phan_loai, StringType())
df2.withColumn("loai_don",
               phan_loai_udf(F.col("thanh_tien"))).show()

# Dang ky them ten ham de goi trong cau lenh SQL
spark.udf.register("PHAN_LOAI", phan_loai, StringType())
spark.sql("""
    SELECT ma_don, thanh_tien, PHAN_LOAI(thanh_tien) AS loai_don
    FROM don_hang
""").show()

## Đoạn mã 4.14. Pandas UDF vector hóa trao đổi dữ liệu qua Apache Arrow.


In [ ]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf("double")
def gia_sau_chiet_khau(thanh_tien: pd.Series) -> pd.Series:
    # Xu ly vector hoa tren tung lo du lieu, khong lap tung hang
    return thanh_tien * 0.95

df2.withColumn("sau_chiet_khau",
               gia_sau_chiet_khau("thanh_tien")).show()

## Đoạn mã 4.15. Thay thế UDF bằng hàm dựng sẵn when/otherwise.


In [ ]:
df2.withColumn(
    "loai_don",
    F.when(F.col("thanh_tien").isNull(), "Khong xac dinh")
     .when(F.col("thanh_tien") >= 1000, "Don lon")
     .otherwise("Don thuong")).show()